In [1]:
import numpy as np
from aeon.datasets import load_regression
from aeon.regression.convolution_based import MultiRocketRegressor, RocketRegressor
from sklearn.metrics import mean_squared_error, r2_score
from tscglue.models import TSCGlueRegressor

In [2]:
DATASET = "FloodModeling1"

X_train, y_train = load_regression(DATASET, split="train")
X_test, y_test = load_regression(DATASET, split="test")

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"y range: [{y_train.min():.3f}, {y_train.max():.3f}]")

Train: (471, 1, 266), Test: (202, 1, 266)
y range: [0.405, 0.675]


/tmp/ipykernel_3950632/2066966295.py:3: FutureWarning: Call to deprecated function (or staticmethod) load_regression. (load_regression parameters load_equal_length and load_no_missing will default to False in version 1.5.0) -- Deprecated since version 1.4.0.
  X_train, y_train = load_regression(DATASET, split="train")
/tmp/ipykernel_3950632/2066966295.py:4: FutureWarning: Call to deprecated function (or staticmethod) load_regression. (load_regression parameters load_equal_length and load_no_missing will default to False in version 1.5.0) -- Deprecated since version 1.4.0.
  X_test, y_test = load_regression(DATASET, split="test")


In [3]:
def add_noise(X, rng, scale=1e-6):
    """Add tiny noise to flat series so variance checks pass."""
    return X + rng.normal(0, scale, X.shape)

def evaluate(name, model, X_train, y_train, X_test, y_test):
    rng = np.random.default_rng()
    X_train = add_noise(X_train, rng)
    X_test = add_noise(X_test, rng)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name:30s}  RMSE={rmse:.4f}  R2={r2:.4f}")
    return {"model": name, "rmse": rmse, "r2": r2}

In [4]:
results = []

results.append(evaluate(
    "RocketRegressor",
    RocketRegressor(),
    X_train, y_train, X_test, y_test,
))

results.append(evaluate(
    "MultiRocketRegressor",
    MultiRocketRegressor(),
    X_train, y_train, X_test, y_test,
))

results.append(evaluate(
    "TSCGlueRegressor",
    TSCGlueRegressor(verbose=10, k_folds=10, n_jobs=20),
    X_train, y_train, X_test, y_test,
))

RocketRegressor                 RMSE=0.0139  R2=0.4532
MultiRocketRegressor            RMSE=0.0044  R2=0.9450
[subprocess] fit_transform quant: 0.2537s
[3.63s] Fit+transformed quant features (471, 2481) (8.92 MB) dtype=float64 in 3.6257s
[9.49s] Fit+transformed rstsf-random_s_759616150 features (471, 13027) (46.81 MB) dtype=float64 in 5.8587s
StratifiedKFold failed, falling back to regular KFold with n_splits=10
StratifiedKFold failed, falling back to regular KFold with n_splits=10
[17.52s] Trained rstsf-random-etr_s_1416886127 in 3.3671s for f-0
[17.79s] Trained rstsf-random-etr_s_1416886127 in 3.4469s for f-3
[17.83s] Trained rstsf-random-etr_s_1416886127 in 3.4162s for f-5
[17.85s] Trained rstsf-random-etr_s_1416886127 in 3.5473s for f-2
[17.92s] Trained rstsf-random-etr_s_1416886127 in 3.5259s for f-4
[17.94s] Trained rstsf-random-etr_s_1416886127 in 3.4503s for f-8
[17.95s] Trained rstsf-random-etr_s_1416886127 in 3.5283s for f-7
[17.97s] Trained rstsf-random-etr_s_1416886127 in 3

In [5]:
import polars as pl

pl.DataFrame(results).sort("rmse")

model,rmse,r2
str,f64,f64
"""MultiRocketRegressor""",0.004418,0.945047
"""TSCGlueRegressor""",0.005368,0.918886
"""RocketRegressor""",0.013937,0.453204
